# A3.9 · Turning a control off without turning the system into an experiment

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.8 · Shared infrastructure between agent runs](https://spbreed.github.io/cyber-commons/lessons/A3.8.html)**.

| | |
|---|---|
| Tools used | OPA, Kyverno |

## What this lesson is

**What it covers.** Disable a classifier without an approved exemption and watch the platform refuse; then launch 50,000 agents under one that is approved.

**Why a security engineer needs it.** Classifiers off, no compensating cap, and tens of thousands of agents launched under that configuration. Each decision was defensible; the combination was never evaluated. The control it builds is: an exemption gate the platform enforces (C6.1), caps that tighten as coverage drops (C6.2), quotas on population and lifetime (C8.2), and an exemption register reviewed at fleet-launch approval (C6.4).

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Turning the cyber classifiers off for a cyber evaluation was defensible. Launching tens of thousands of agents was a normal request. Nothing in the path required the two decisions to be considered in the same sentence, and the combination was never evaluated by anyone.

> **At CyberTravels.** Alex turned the guardrails off for the demo. That was defensible. What was never decided is what CyberTravels' blast radius should have shrunk to while they were off. R2.

## 2 · The framework

```
   coverage                    permitted blast radius

   all controls on             50,000 agents . 72h . allowlist+
   one detective off              200 agents .  8h . allowlist
   two or more off                 25 agents .  2h . allowlist

        exemption record ----> orchestrator ----> refuse or terminate
        (control, scope, expiry,        (caps are charged, not remembered)
         compensating control)
```

Controls get disabled. A classifier blocks the evaluation you are trying to
run; an egress rule breaks a legitimate integration; a rate limit makes a test
impossible. Pretending otherwise produces shadow exemptions rather than fewer
of them.

What makes an exemption safe is not the approval. It is that **reduced
detective coverage automatically buys a smaller blast radius**, enforced
somewhere the workload cannot reach.

In the source incident that link was absent. Cyber classifiers were turned off
for a cyber evaluation — defensible on its own — and tens of thousands of
agents were then launched under that configuration, with no compensating cap.
The report assesses the classifiers would likely have blocked many of the
attack actions. Each decision was reasonable; the *combination* was never
evaluated, because nothing forced the two to be considered together.

Four controls:

**The exemption gate (C6.1).** A written record naming the control, the scope,
the justification, the compensating control and a hard expiry — and it must be
technically enforced. If the platform cannot express the exemption, it cannot
be granted.

**Blast-radius caps (C6.2).** Egress allowlist, concurrency, runtime, spend and
target scope, defaulting tighter as coverage decreases, enforced in the
orchestrator and terminating on breach rather than alerting.

**Population and lifetime quotas (C8.2)**, tied to the exemption state so
reduced monitoring lowers them automatically.

**A register reviewed at launch (C6.4).** Scale is the variable that changes
the risk: the same exemption is fine at 100 agents and unacceptable at 50,000.

## 3 · The check, as a skill

Alex turned the guardrails off for the demo, and that was defensible. The skill checks what made it defensible — a bounded exemption the platform can read — and then decides the same launch twice: control by control, and against the exemption class. The two answers disagree, and that is the finding.

In [ ]:
# skills/runtime/control-exemption-audit/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: control-exemption-audit
description: >-
  Check that turning a control off is a recorded, bounded exemption the platform
  reads, and that risky launches are decided against the exemption class rather
  than one control at a time. Use when guardrails are disabled for a demo, a
  migration, or a load test.
allowed-tools: Read, Grep, Glob
---

# Turning a control off is defensible; not recording it is not

Every estate disables controls sometimes. The failure is not the disabling — it
is that nothing recorded it, nothing bounded it, and the next decision was made
by a system that believed the control was on. And decisions made one control at
a time approve launches that the same facts refuse when taken together.

## When to use this

Reviewing guardrail configuration, change management for agent platforms, or
any launch where "we turned that off for the demo" appears in a thread.

## Procedure

**1 — Compare intended state with actual state,** per control. The gap is the
set of undeclared exemptions, and each one is a finding regardless of whether
it was reasonable.

**2 — Require the exemption to be a record the platform reads.** Control,
requester, approver, reason, expiry. A record in a ticket the platform cannot
read is documentation; the orchestrator must refuse a disable with no matching
record.

**3 — Test both directions.** A named control with a valid exemption may be
disabled. A control with no approval must be refused. Both, or it is a logging
system.

**4 — Decide a launch one control at a time.** Take a large launch and evaluate
each control's own rule. It will be approved, and each answer will be correct in
isolation. Record that.

**5 — Re-decide against the exemption class.** Group the disabled controls —
detective, preventive, egress — and set caps per class: how many agents, for how
long, with what compensating control. The same launch is now refused, and a
smaller one on an allowlist is permitted. The disagreement between step 4 and
step 5 is the finding.

## Output contract

```json
{
  "controls": [{"name": "str", "intended": "on|off", "actual": "on|off", "exemption": "str|null"}],
  "exemptions": [{"control": "str", "approver": "str", "expires": "str", "platform_readable": true}],
  "probes": {"named_with_approval": "permitted", "unapproved": "refused"},
  "launch": {"request": {"agents": 0, "hours": 0}, "per_control_verdict": "approved",
             "by_exemption_class_verdict": "refused", "caps": {"class": "str", "agents": 0, "hours": 0}}
}
```

## Failure modes

- **An exemption the platform cannot read.** It cannot enforce an expiry it
  cannot see.
- **Deciding control by control.** Each answer is right and the composition is
  wrong.
- **No expiry.** A permanent exemption is a policy change made by whoever was
  on call.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/runtime/control-exemption-audit/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/runtime/control-exemption-audit/scripts/control_exemption_audit.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Decide a launch one control at a time and then against an exemption class, and watch the two answers disagree.

This is the executable half of the `control-exemption-audit` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

EXEMPTIONS = {
 "EX-118": {"control": "cyber-classifier", "scope": "exploitgym-eval",
            "justification": "classifier blocks the behaviour under test",
            "compensating": "egress allowlist + concurrency cap",
            "expires_day": 30, "approved_by": "security"},
}
CONTROL_STATE = {"cyber-classifier": "disabled", "egress-allowlist": "enabled",
                 "transcript-signing": "enabled"}

def may_disable(control, day, exemptions):
    for eid, x in sorted(exemptions.items()):
        if x["control"] == control and day <= x["expires_day"]:
            return True, eid
    return False, None

for control, day in (("cyber-classifier", 12), ("cyber-classifier", 44),
                     ("transcript-signing", 12)):
    ok, eid = may_disable(control, day, EXEMPTIONS)
    print(f"day {day:>3}  disable {control:20s}"
          f"{'ALLOW via ' + eid if ok else 'REFUSE - no approved exemption'}")
print()
print("The record is the mechanism, not a wiki page describing one. An")
print("exemption the platform cannot express is an exemption you cannot grant.")
assert may_disable("cyber-classifier", 12, EXEMPTIONS)[0]
assert not may_disable("transcript-signing", 12, EXEMPTIONS)[0]

LAUNCH = {"agents": 50000, "runtime_hours": 72, "egress": "any",
          "exemptions": ["EX-118"]}

def naive_launch(request):
    """Each decision checked alone, which is how the incident's was."""
    ok, _ = may_disable("cyber-classifier", 12, EXEMPTIONS)
    return {"exemption_valid": ok, "launched": True, "agents": request["agents"]}

r = naive_launch(LAUNCH)
print("checked one decision at a time:")
print(f"   exemption valid : {r['exemption_valid']}")
print(f"   launch approved : {r['launched']}  ({r['agents']:,} agents)")
print()
print("Both answers are correct. Turning the classifier off was justified and")
print("approved; launching at scale was a normal request. Nothing in the path")
print("required the two to be considered in the same sentence, and the report")
print("assesses the classifier would likely have blocked many of the actions")
print("that followed.")
assert r["launched"] and r["agents"] == 50000

CAPS = {                       # exemption class -> caps
 "none":              {"agents": 50000, "runtime_hours": 72, "egress": "allowlist+"},
 "one-detective-off": {"agents": 200,   "runtime_hours": 8,  "egress": "allowlist"},
 "two-or-more-off":   {"agents": 25,    "runtime_hours": 2,  "egress": "allowlist"},
}
DETECTIVE = {"cyber-classifier", "egress-allowlist", "transcript-signing"}

def exemption_class(state):
    off = sum(1 for c, s in state.items() if c in DETECTIVE and s != "enabled")
    return "none" if off == 0 else "one-detective-off" if off == 1 else "two-or-more-off"

def orchestrate(request, state):
    cls = exemption_class(state)
    caps = CAPS[cls]
    breaches = [k for k in ("agents", "runtime_hours")
                if request[k] > caps[k]]
    if request["egress"] != "allowlist" and cls != "none":
        breaches.append("egress")
    return {"class": cls, "caps": caps, "breaches": breaches,
            "launched": not breaches}

for label, state in (("all controls on", {"cyber-classifier": "enabled",
                                          "egress-allowlist": "enabled",
                                          "transcript-signing": "enabled"}),
                     ("classifier off", CONTROL_STATE)):
    out = orchestrate(LAUNCH, state)
    print(f"{label:18s}class={out['class']:18s}"
          f"cap={out['caps']['agents']:>6,} agents  "
          f"{'LAUNCH' if out['launched'] else 'REFUSED: ' + ','.join(out['breaches'])}")

ok = orchestrate({"agents": 200, "runtime_hours": 8, "egress": "allowlist"},
                 CONTROL_STATE)
print(f"\nsame exemption, 200 agents for 8h on an allowlist: "
      f"{'LAUNCH' if ok['launched'] else 'refused'}")
print()
print("The exemption did not become unavailable. Its price became visible, and")
print("the orchestrator charges it rather than asking someone to remember.")
assert not orchestrate(LAUNCH, CONTROL_STATE)["launched"] and ok["launched"]

SCENARIOS = [
 ("classifiers on",                {"cyber-classifier": "enabled",
                                    "egress-allowlist": "enabled",
                                    "transcript-signing": "enabled"}),
 ("classifier off",                {"cyber-classifier": "disabled",
                                    "egress-allowlist": "enabled",
                                    "transcript-signing": "enabled"}),
 ("classifier + signing off",      {"cyber-classifier": "disabled",
                                    "egress-allowlist": "enabled",
                                    "transcript-signing": "disabled"}),
]
print(f"{'state':28s}{'class':20s}{'agent cap':>10}{'runtime':>9}")
for label, state in SCENARIOS:
    cls = exemption_class(state)
    caps = CAPS[cls]
    print(f"{label:28s}{cls:20s}{caps['agents']:>10,}{caps['runtime_hours']:>8}h")

print()
print("50,000 -> 200 -> 25. The second exemption costs an order of magnitude")
print("more than the first, which is the correct shape: compounding is where")
print("the incident's risk sat, and it is the case a human reviewer is worst at")
print("holding in their head.")
assert CAPS["two-or-more-off"]["agents"] < CAPS["one-detective-off"]["agents"]

## What you just proved

An exemption record the platform reads permits disabling one named control and refuses another with no approval. Checked one decision at a time, a 50,000-agent launch with classifiers off is approved — both answers correct in isolation. Tied to exemption class, the same launch is refused and 200 agents for 8 hours on an allowlist is permitted; a second exemption drops the cap again, from 200 to 25.

## Your turn

Find one control currently disabled in your environment and ask three questions: who approved it, when does it expire, and what cap came with it. The third question is the one that usually has no answer.

---

**Next → [A3.10 · The agent's escalation path](https://spbreed.github.io/cyber-commons/lessons/A3.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*